# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


In [ ]:
import sys
from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from pyspark.sql import functions as F

# Inicialización
args = getResolvedOptions(sys.argv, ['JOB_NAME'])
sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
job.init(args['JOB_NAME'], args)

# 1. Carga de datos (Bronze)
path_bronze = "s3://proyecto-ny311/bronze/311-service-requests-from-2010-to-present.csv"
df = spark.read.csv(path_bronze, header=True, inferSchema=True)

# 2. Conteo de Duplicados Totales (sobre todo el DF)
total_count = df.count()
distinct_count = df.distinct().count()
duplicados_totales = total_count - distinct_count

# 3. Healthcheck por columna
# Identificar columnas numéricas para Outliers
num_cols = [f.name for f in df.schema.fields if str(f.dataType) in ["IntegerType", "DoubleType", "FloatType", "LongType"]]

# Lista para recolectar métricas
metrics_list = []

for col_name in df.columns:
    # Métricas básicas
    null_count = df.filter(F.col(col_name).isNull() | (F.col(col_name) == "") | (F.col(col_name) == "Unspecified")).count()
    unique_count = df.select(col_name).distinct().count()
    
    # Inicializar valores de outliers
    out_n, lower, upper = 0, 0.0, 0.0
    
    if col_name in num_cols:
        # Usamos approxQuantile para eficiencia (error 0.01)
        # Esto es vital para 25 millones de datos
        quantiles = df.stat.approxQuantile(col_name, [0.25, 0.75], 0.01)
        if len(quantiles) == 2:
            q1, q3 = quantiles[0], quantiles[1]
            iqr = q3 - q1
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            out_n = df.filter((F.col(col_name) < lower) | (F.col(col_name) > upper)).count()

    metrics_list.append((
        col_name,
        str(df.schema[col_name].dataType),
        null_count,
        round((null_count / total_count) * 100, 2),
        unique_count,
        out_n,
        round((out_n / total_count) * 100, 2),
        float(lower),
        float(upper)
    ))

# 4. Crear DataFrame de resultados
schema_metrics = ["columna", "tipo_dato", "nulos_n", "nulos_pct", "unicos_n", "outliers_n", "outliers_pct", "iqr_lower", "iqr_upper"]
df_health = spark.createDataFrame(metrics_list, schema=schema_metrics)

# Añadir metadatos globales
df_health = df_health.withColumn("total_filas_dataset", F.lit(total_count)) \
                     .withColumn("duplicados_dataset", F.lit(duplicados_totales))

# 5. Guardar reporte en S3 (Metadata Layer)
# Usamos .coalesce(1) para que Streamlit lea un solo archivo CSV pequeño
output_path = "s3://proyecto-ny311/metadata/healthcheck_report/"
df_health.coalesce(1).write.mode("overwrite").option("header", "true").csv(output_path)

job.commit()